# ⚒️ Laboratorio Metalúrgico - Lenovo LOQ
**Conectado a:** `G:\Mi unidad\02_Trabajo_Metalurgia`  
Presiona **`Shift + Enter`** en cada celda para ejecutar los cálculos y ver las gráficas interactivas.

In [ ]:
# CELDA 1: MÓDULO DE CONMINUCIÓN Y GRANULOMETRÍA (Cálculo de P80 y Modelo Rosin-Rammler)
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from pathlib import Path

 mallas = ["+100", "+140", "+200", "+270", "+400", "-400"]
abertura_um = np.array([150.0, 106.0, 75.0, 53.0, 38.0, 0.0])
peso_retenido_g = np.array([45.2, 82.5, 110.4, 68.0, 54.3, 139.6])

df_gran = pd.DataFrame({"Malla Tyler": mallas, "Abertura (um)": abertura_um, "Peso (g)": peso_retenido_g})
df_gran["% Retenido"] = (df_gran["Peso (g)"] / df_gran["Peso (g)"].sum()) * 100
df_gran["% Ret. Acumulado"] = df_gran["% Retenido"].cumsum()
df_gran["% Pasante Acum. F(x)"] = 100 - df_gran["% Ret. Acumulado"]

# Ajuste Rosin-Rammler: F(x) = 100 * (1 - exp(-(x / d63.2)^n))
x_data = df_gran.loc[:4, "Abertura (um)"].values
y_data = df_gran.loc[:4, "% Pasante Acum. F(x)"].values

def rosin_rammler(x, d63, n):
    return 100.0 * (1.0 - np.exp(-((x / d63) ** n)))

params, _ = curve_fit(rosin_rammler, x_data, y_data, p0=[100.0, 1.2])
d63, n_rr = params
P80 = d63 * ((-np.log(1.0 - 0.80)) ** (1.0 / n_rr))

print(f"✅ P80 Calculado (Rosin-Rammler): {P80:.2f} micrones (µm)")
display(df_gran.round(2))

# Gráfica interactiva con Plotly
x_curve = np.linspace(10, 200, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name='Datos Laboratorio', marker=dict(size=10)))
fig.add_trace(go.Scatter(x=x_curve, y=rosin_rammler(x_curve, d63, n_rr), mode='lines', name='Ajuste Rosin-Rammler'))
fig.add_hline(y=80, line_dash='dash', line_color='red', annotation_text=f'P80 = {P80:.1f} µm')
fig.update_layout(title='Curva Granulométrica de Molienda', xaxis_title='Abertura (µm) - Escala Log', yaxis_title='% Pasante Acumulado', xaxis_type='log', template='plotly_dark')
fig.show()

In [ ]:
# CELDA 2: BALANCE METALÚRGICO DE 2 PRODUCTOS Y EXPORTACIÓN A GOOGLE DRIVE (G:)
F = 2400.0   # Toneladas Métricas Secas de Alimentación (TMS/día)
f = 1.18     # Ley de Cabeza (% Cu)
c = 25.40    # Ley de Concentrado (% Cu)
t = 0.11     # Ley de Relave (% Cu)

C = F * (f - t) / (c - t)                 # Tonelaje Concentrado
T = F - C                                 # Tonelaje Relave
R = (c * (f - t)) / (f * (c - t)) * 100   # Recuperación Metalúrgica (%)
RC = (c - t) / (f - t)                    # Ratio de Concentración

df_bal = pd.DataFrame({
    "Flujo": ["Alimentación (Cabeza)", "Concentrado Final", "Relave General"],
    "Peso (TMS)": [F, C, T],
    "% Peso": [100.0, (C/F)*100, (T/F)*100],
    "Ley Cu (%)": [f, c, t],
    "Fino Cu (TM)": [F*f/100, C*c/100, T*t/100],
    "Distribución Cu (%)": [100.0, R, 100.0 - R]
})

ruta_drive = Path(r"G:\Mi unidad\02_Trabajo_Metalurgia\01_Balances_y_Planta")
archivo_reporte = ruta_drive / "Reporte_Planta_P80_y_Balance.xlsx"

with pd.ExcelWriter(archivo_reporte, engine="xlsxwriter") as writer:
    df_bal.round(2).to_excel(writer, sheet_name="Balance_Flotacion", index=False)
    df_gran.round(2).to_excel(writer, sheet_name="Granulometria_P80", index=False)

print(f"✅ Recuperación Cu: {R:.2f}% | Ratio de Concentración: {RC:.2f} : 1")
print(f"📁 Reporte con 2 pestañas guardado en tu Drive: {archivo_reporte}")
display(df_bal.round(2))

: 